In [ ]:
from py123d.api import SceneFilter

In [ ]:
from py123d.api import get_filtered_scenes

scene_filter = SceneFilter(
    datasets=["nuplan-mini"],
    split_names=None,
    log_names=None,
    scene_uuids=None,
    target_iteration_duration_s=0.1,  # 10Hz iteration frequency
    future_duration_s=10.0,  # Look up to 1 second into the future.
    history_duration_s=5.0,  # Look up to 0.5 seconds into the past.
    required_scene_modalities=["ego_state_se3", "lidar.lidar_merged"],
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
# Init PDM-Closed

from nav123d.pdm.pdm_closed_planner import get_pdm_closed_planner

pdm_closed_planner = get_pdm_closed_planner()

In [ ]:
scene = scenes[0]

iteration = 0


modality = scene.get_custom_modality_at_iteration(iteration, "scenario")
assert modality is not None, "Scenario modality not found at iteration 2."
lane_group_ids = [int(id_) for id_ in modality.data["route_roadblock_ids"]]
lane_group_ids

In [ ]:
from nav123d.pdm.pdm_closed_planner import PDMClosedInput

ego_state_se3 = scene.get_ego_state_se3_at_iteration(iteration)
box_detections_se3 = scene.get_box_detections_se3_at_iteration(iteration)
traffic_light_detections = scene.get_traffic_light_detections_at_iteration(iteration)


assert ego_state_se3 is not None, "Ego state modality not found at iteration."
assert box_detections_se3 is not None, "Box detections modality not found at iteration."
assert traffic_light_detections is not None, "Traffic light detections modality not found at iteration."

planner_input = PDMClosedInput(
    ego_state_se2=ego_state_se3.ego_state_se2,
    box_detections_se2=box_detections_se3.box_detections_se2,
    traffic_light_detections=traffic_light_detections,
)

In [ ]:
map_api = scene.get_map_api()
assert map_api is not None, "MapAPI is not available in the scene."
pdm_closed_planner.initialize(map_api, lane_group_ids)

In [ ]:
trajectory = pdm_closed_planner.compute_planner_trajectory(planner_input)

In [ ]:
import matplotlib.pyplot as plt
from py123d.visualization.matplotlib.observation import add_scene_on_ax

fig, ax = plt.subplots(figsize=(10, 10))

add_scene_on_ax(ax, scene, iteration, radius=20.0)